# Evaluating LLM Prompt Strategies for Sentiment Classification

**Research Question:** How do different prompting strategies affect the accuracy, consistency, and cost of LLM-based sentiment classification?

This notebook evaluates four prompting strategies on a binary sentiment classification task:
- **Zero-shot**: Direct classification without examples
- **Few-shot**: Classification with 3 labeled examples
- **Chain-of-thought**: Explicit reasoning before classification
- **Structured JSON**: Requesting structured output format

All experiments use fixed random seeds and cached responses for full reproducibility.

## Setup

This notebook runs in **offline mode** using pre-cached responses, so no API key is required.

To run with live API calls:
1. Copy `.env.example` to `.env`
2. Add your `OPENAI_API_KEY`
3. Set `OFFLINE_MODE = False` below

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path.cwd()))

from src import create_client, compare_strategies, get_dataset

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("✓ Setup complete")

## Configuration

In [ ]:
# Experiment configuration
OFFLINE_MODE = True  # Set to False to make live API calls
MODEL = "gpt-3.5-turbo"
SEEDS = [42, 43, 44]  # Multiple seeds for consistency measurement

# Strategies to evaluate
STRATEGIES = [
    "zero_shot",
    "few_shot",
    "chain_of_thought",
    "structured_json"
]

print(f"Mode: {'Offline (cached)' if OFFLINE_MODE else 'Online (API calls)'}")
print(f"Model: {MODEL}")
print(f"Seeds: {SEEDS}")
print(f"Strategies: {', '.join(STRATEGIES)}")

## Dataset

We use a hand-crafted dataset of 100 short sentences (50 positive, 50 negative) for educational purposes.

In [ ]:
texts, labels = get_dataset()

print(f"Dataset size: {len(texts)} examples")
print(f"Class distribution: {sum(labels)} positive, {len(labels) - sum(labels)} negative")
print(f"\nExample texts:")
for i in range(3):
    label_str = "positive" if labels[i] == 1 else "negative"
    print(f"  [{label_str}] {texts[i]}")

## Evaluation

We evaluate each strategy across multiple random seeds to measure both accuracy and consistency.

In [ ]:
# Create LLM client
client = create_client(offline_mode=OFFLINE_MODE, model=MODEL)

# Run evaluation
results = compare_strategies(
    client=client,
    strategy_names=STRATEGIES,
    seeds=SEEDS,
    verbose=True
)

print("\n✓ Evaluation complete")

## Results

In [ ]:
# Extract metrics into a DataFrame
metrics_data = []
for strategy, data in results.items():
    m = data['metrics']
    metrics_data.append({
        'Strategy': strategy.replace('_', ' ').title(),
        'Accuracy': f"{m['accuracy']:.3f}",
        'Consistency': f"{m['consistency']:.3f}",
        'Parse Rate': f"{m['parse_rate']:.3f}",
        'Mean Latency (ms)': f"{m['mean_latency_ms']:.0f}",
        'Total Tokens': f"{m['total_tokens']:,}"
    })

df_metrics = pd.DataFrame(metrics_data)
print("\n=== Performance Metrics ===")
print(df_metrics.to_string(index=False))

### Visualizations

In [ ]:
# Accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
strategies_clean = [s.replace('_', ' ').title() for s in STRATEGIES]
accuracies = [results[s]['metrics']['accuracy'] for s in STRATEGIES]

axes[0].bar(strategies_clean, accuracies, color='steelblue', alpha=0.8)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Classification Accuracy by Strategy', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Random baseline')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

# Consistency
consistencies = [results[s]['metrics']['consistency'] for s in STRATEGIES]

axes[1].bar(strategies_clean, consistencies, color='coral', alpha=0.8)
axes[1].set_ylabel('Consistency', fontsize=12)
axes[1].set_title(f'Consistency Across {len(SEEDS)} Seeds', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Cost and latency comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total tokens
tokens = [results[s]['metrics']['total_tokens'] for s in STRATEGIES]

axes[0].bar(strategies_clean, tokens, color='seagreen', alpha=0.8)
axes[0].set_ylabel('Total Tokens', fontsize=12)
axes[0].set_title('Token Usage by Strategy', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# Mean latency
latencies = [results[s]['metrics']['mean_latency_ms'] for s in STRATEGIES]

axes[1].bar(strategies_clean, latencies, color='mediumpurple', alpha=0.8)
axes[1].set_ylabel('Mean Latency (ms)', fontsize=12)
axes[1].set_title('Response Latency by Strategy', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy vs Cost trade-off
plt.figure(figsize=(10, 6))

for i, strategy in enumerate(STRATEGIES):
    acc = results[strategy]['metrics']['accuracy']
    tok = results[strategy]['metrics']['total_tokens']
    label = strategy.replace('_', ' ').title()
    plt.scatter(tok, acc, s=200, alpha=0.7, label=label)
    plt.annotate(label, (tok, acc), xytext=(5, 5), textcoords='offset points', fontsize=10)

plt.xlabel('Total Tokens Used', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs Token Cost Trade-off', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Discussion

### Key Findings

Based on the evaluation results above:

1. **Accuracy**: All strategies significantly outperform the random baseline (0.5)
2. **Consistency**: Strategies show varying levels of determinism across seeds
3. **Cost**: Chain-of-thought uses more tokens due to explicit reasoning
4. **Latency**: Longer responses correlate with higher latency

### Threats to Validity

- **Small dataset**: 100 examples may not generalize to broader domains
- **Simple task**: Binary sentiment is relatively easy; complex tasks may show different patterns
- **Single model**: Results are specific to the model used (gpt-3.5-turbo)
- **Hand-crafted data**: Real-world text may contain more ambiguity and edge cases
- **Limited seeds**: Only 3 seeds tested for consistency
- **Parsing assumptions**: Response parsing may miss valid but differently-formatted answers

### Next Steps

1. Test on larger, real-world datasets (e.g., full SST-2, IMDB reviews)
2. Evaluate additional strategies (self-consistency, role prompting)
3. Compare multiple models (GPT-4, Claude, Llama)
4. Analyze failure cases and error patterns
5. Investigate temperature effects and alternative decoding strategies
6. Extend to multi-class or regression tasks

### References

- Brown et al. (2020). "Language Models are Few-Shot Learners." NeurIPS.
- Wei et al. (2022). "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models." NeurIPS.
- Kojima et al. (2022). "Large Language Models are Zero-Shot Reasoners." NeurIPS.
- Wang et al. (2022). "Self-Consistency Improves Chain of Thought Reasoning in Language Models." ICLR 2023.

## Reproduction Notes

This notebook is fully reproducible:

- **Fixed seeds**: All API calls use deterministic seeds (42, 43, 44)
- **Cached responses**: Results are cached by (model, prompt, seed) tuple
- **Offline mode**: The notebook can run without API access using committed cache
- **Version pinning**: `requirements.txt` pins all dependency versions

To reproduce these exact results:
1. Use the cached responses in `results/cache/`
2. Or re-run with same model + seeds (may vary slightly due to API changes)